# Multi-Agent Patterns

This notebook teaches you 12 ways to organize multiple AI agents working together.

**Multi-agent** means more than one AI agent working on a task. Each agent has its own role and instructions. They communicate by passing messages.

**Why multiple agents?** One agent doing everything gets confused. Focused specialists produce better results -- like a team of experts.

By the end, you will know: Supervisor, Hierarchical, Swarm, Debate, Collaborative, Specialist Routing, Consensus, Assembly Line, Mixture of Agents, Society of Mind, AutoGen-style, and CrewAI-style patterns.

## Setup

In [1]:
!pip install python_dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

In [3]:
from openai import OpenAI
import json
import time

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

**What happened:** We imported the OpenAI library and set up our client.

## Helper Function

In [8]:
def ask_agent(agent_name, system_prompt, user_message, max_tokens=300):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system", 
                "content": system_prompt
            },
            {
                "role": "user", 
                "content": user_message
            }
        ],
        max_tokens=max_tokens
    )
    
    reply = response.choices[0].message.content
    print("[" + agent_name + "]:", reply[:200])
    print()
    return reply

print("ask_agent helper is ready")

ask_agent helper is ready


**What happened:** We created `ask_agent`, a helper that takes an agent name, system prompt, and user message, calls the model, and prints the reply. We use this throughout the notebook.

## 1. Supervisor Pattern

**What it does:** One agent is the **supervisor** (boss). It receives the task, decides which worker agents to call and in what order, and makes the final decision.

**When to use it:** When you need central control over how agents work together.

**Steps:**
1. User gives the task to the supervisor.
2. Supervisor decides which worker to call.
3. Worker does the job and returns the result.
4. Supervisor calls more workers or gives the final answer.

In [25]:
def supervisor_pattern(task):
    
    # Define workers
    workers = {
        "researcher": "You are a researcher. Find facts and information about the topic. Be brief (2-3 sentences).",
        "writer": "You are a writer. Write clear, polished text based on the information given. Be brief (2-3 sentences).",
        "critic": "You are a critic. Review the text and suggest one specific improvement. Be brief (2-3 sentences)."
    }

    
    
    # Supervisor decides the plan
    supervisor_prompt = (
        "You are a supervisor managing a team of workers: researcher, writer, critic. "
        "Given a task, decide the order to call workers. "
        "Reply with ONLY the worker names separated by commas. Example: researcher,writer,critic"
    )



    plan = ask_agent("Supervisor", supervisor_prompt, "Task: " + task, max_tokens=50)
    
    worker_order = plan.strip().split(",")
    print("\n\n ------ REPLY 1 FROM SUPERVISOR - worker_order ------ \n\n", worker_order, "\n\n ------ REPLY 1 FROM SUPERVISOR - worker_order ------ \n\n")

    
    
    
    # Execute each worker in order
    context = "Task: " + task
    print("\n\n ------ CONTEXT END ------ \n\n", context, "\n\n ------ CONTEXT END ------ \n\n")
    
    
    
    for worker_name in worker_order:
        
        worker_name = worker_name.strip().lower()
        print("\n\n ------ WORKER NAME END ------ \n\n", worker_name, "\n\n ------ WORKER NAME END ------ \n\n")
        
        
        if worker_name in workers:
            
            system_prompt = workers[worker_name]

            result = ask_agent(worker_name.capitalize(), system_prompt, context)
            
            context = context + "\n\n" + worker_name.capitalize() + " said: " + result

            print("\n\n ------ CONTEXT START: ------ \n\n", context, "\n\n ------ CONTEXT END: ------ \n\n")

    
    
    
    # Supervisor gives final answer

    final = ask_agent(
        "Supervisor",
        "You are a supervisor. Combine your team's work into a final answer. Be brief (3-4 sentences).",
        context
    )
    
    
    return final

**What happened:** We built the Supervisor pattern with a supervisor that decides the order and 3 workers (researcher, writer, critic). Workers pass results through the context, and the supervisor makes the final decision.

In [26]:
supervisor_pattern("Explain why sleep is important for programmers")

[Supervisor]: researcher,writer,critic



 ------ REPLY 1 FROM SUPERVISOR - worker_order ------ 

 ['researcher', 'writer', 'critic'] 

 ------ REPLY 1 FROM SUPERVISOR - worker_order ------ 




 ------ CONTEXT END ------ 

 Task: Explain why sleep is important for programmers 

 ------ CONTEXT END ------ 




 ------ WORKER NAME END ------ 

 researcher 

 ------ WORKER NAME END ------ 


[Researcher]: Sleep is crucial for programmers as it enhances cognitive function, memory consolidation, and problem-solving skills, which are vital for coding and debugging. Lack of sleep can lead to decreased focu



 ------ CONTEXT START: ------ 

 Task: Explain why sleep is important for programmers

Researcher said: Sleep is crucial for programmers as it enhances cognitive function, memory consolidation, and problem-solving skills, which are vital for coding and debugging. Lack of sleep can lead to decreased focus, increased errors, and slower productivity, ultimately affecting the quality of the

'Sleep is vital for programmers as it significantly boosts cognitive function, memory consolidation, and problem-solving skills essential for effective coding and debugging. Insufficient sleep can lead to decreased focus, increased errors, and lower productivity, thus compromising software quality. Additionally, quality sleep supports mental health, reducing stress and burnout in the demanding programming field. Incorporating specific studies could further strengthen the argument about the importance of sleep.'

**What happened:** The supervisor called researcher, writer, and critic in order. Each added to the context, and the supervisor combined everything into a final answer.

## 2. Hierarchical Multi-Agent

**What it does:** Like the supervisor pattern, but with **multiple levels**. A top supervisor delegates to area leads, who each manage their own workers. It forms a tree structure.

**When to use it:** For very complex tasks that need to be broken into major areas, each with sub-tasks.

**Steps:**
1. Top supervisor breaks the task into major areas.
2. Each area lead creates sub-tasks for workers.
3. Workers complete sub-tasks.
4. Area leads summarize; top supervisor combines everything.

In [27]:
def hierarchical_pattern(task):

    # Top-level supervisor breaks task into areas
    areas_prompt = (
        "You manage a team. Break this task into exactly 2 areas. "
        "Reply with ONLY the two area names separated by a comma. "
        "Example: technical,business"
    )

    areas_text = ask_agent("Top Supervisor", areas_prompt, task, max_tokens=30)
    areas = areas_text.strip().split(",")
    print("\n\n ------ AREAS START: ------ \n\n", areas, "\n\n ------ AREAS END: ------ \n\n")

    
    
    all_results = []

    for area in areas:
        area = area.strip()
        print("\n\n ------ START: INSIDE AREA : ------ \n\n", area, "\n\n ------ END: INSIDE AREA ------ \n\n")

        
        # Mid-level supervisor for this area
        mid_prompt = (
            "You are the " + area + " team lead. "
            "You have two workers. Give each worker a specific sub-task for: " + task + " "
            "Reply with ONLY the two sub-tasks separated by |. "
            "Example: research market trends|analyze competitor data"
        )
        subtasks_text = ask_agent(area + " Lead", mid_prompt, task, max_tokens=60)
        subtasks = subtasks_text.strip().split("|")
        print("\n\n ------ EACH AREA - SUBTASKS START: ------ \n\n", subtasks, "\n\n ------ EACH AREA - SUBTASKS END: ------ \n\n")

        

        
        # Workers do their sub-tasks
        area_results = []
        
        for i in range(len(subtasks)):
            
            subtask = subtasks[i].strip()
            
            worker_result = ask_agent(
                area + " Worker " + str(i + 1),
                "You are a worker. Complete this sub-task in 2-3 sentences.",
                subtask
            )

            area_results.append(worker_result)

        print("\n\n ------ EACH AREA - SUBTASK - WORKER RESULTS START: ------ \n\n", area_results, "\n\n ------ EACH AREA - SUBTASK - WORKER RESULT END: ------ \n\n")

        
        


        # Mid-level supervisor combines worker results
        combined_input = "Sub-task results:\n"
        for result in area_results:
            combined_input = combined_input + "- " + result + "\n"

        print("\n\n ------ EACH AREA - SUBTASK - WORKER - ALL WORKERS COMBINED RESULTS START: ------ \n\n", combined_input, "\n\n ------ EACH AREA - SUBTASK - WORKER - ALL WORKERS COMBINED RESULTS END: ------ \n\n")

        
        
        area_summary = ask_agent(
            area + " Lead",
            "Summarize your team's work in 2-3 sentences.",
            combined_input
        )
        all_results.append(area + ": " + area_summary)
        
        print("\n\n ------ EACH AREA - SUBTASK - WORKER - FINAL RESULTS FROM LLM START: ------ \n\n", all_results, "\n\n ------ EACH AREA - SUBTASK - WORKER - FINAL RESULTS FROM LLM START: ------ \n\n")

    

    
    # Top supervisor combines all areas
    final_input = "Results from all areas:\n"
    for result in all_results:
        final_input = final_input + result + "\n\n"

    print("\n\n ------ ALL AREA RESPONSES: MERGED  ------ \n\n", final_input, "\n\n ------ ALL AREA RESPONSES: MERGED ------ \n\n")

    
    
    final = ask_agent(
        "Top Supervisor",
        "Combine all team results into a final answer. Be brief (3-4 sentences).",
        final_input
    )
    
    
    print("\n\n ------ FINAL RESPONSE  ------ \n\n", final_input, "\n\n ------ FINAL RESPONSE ------ \n\n")

    return final

**What happened:** We built a 3-level hierarchy: top supervisor breaks the task into areas, area leads assign sub-tasks to workers, and results flow up. This is like CEO -> department heads -> workers.

In [28]:
hierarchical_pattern("Create a plan for launching a new mobile app")

[Top Supervisor]: marketing,development



 ------ AREAS START: ------ 

 ['marketing', 'development'] 

 ------ AREAS END: ------ 




 ------ START: INSIDE AREA : ------ 

 marketing 

 ------ END: INSIDE AREA ------ 


[marketing Lead]: develop marketing strategy|design social media campaign



 ------ EACH AREA - SUBTASKS START: ------ 

 ['develop marketing strategy', 'design social media campaign'] 

 ------ EACH AREA - SUBTASKS END: ------ 


[marketing Worker 1]: To develop an effective marketing strategy, start by identifying your target audience and understanding their needs and preferences. Next, analyze your competitors to find gaps in the market that you 

[marketing Worker 2]: To design a successful social media campaign, start by defining clear objectives such as increasing brand awareness or driving website traffic. Utilize eye-catching visuals and engaging content tailor



 ------ EACH AREA - SUBTASK - WORKER RESULTS START: ------ 

 ['To develop an effective marketin

'The marketing and development teams have collaboratively crafted a strategic plan to optimize brand promotion and consumer engagement. They focused on identifying target audiences and user personas to tailor marketing efforts, employing a mix of digital and traditional channels. Their structured three-month strategy includes phases for market research, content creation, and ongoing performance evaluations to ensure continuous improvement and effectiveness in reaching campaign objectives.'

**What happened:** The top supervisor split the task into two areas. Each area had its own lead and workers. Results flowed up: workers -> leads -> top supervisor -> final answer.

## 3. Swarm Pattern

**What it does:** Agents work independently with **no central control**. Each agent tackles the problem from a different angle, sees what others said, and responds.

**When to use it:** When you want diverse perspectives without a boss directing traffic. Good for brainstorming.

**Steps:**
1. Multiple agents give initial thoughts independently.
2. Agents see what others said and respond in a second round.
3. A synthesizer combines all perspectives.

In [35]:
def swarm_pattern(task):
    print("=== SWARM PATTERN ===")
    print("Task:", task)
    print()

    
    
    agents = [
        {"name": "Optimist", "prompt": "You always see the positive side. Give your perspective in 2-3 sentences."},
        {"name": "Skeptic", "prompt": "You question everything and look for problems. Give your perspective in 2-3 sentences."},
        {"name": "Pragmatist", "prompt": "You focus on what is practical and actionable. Give your perspective in 2-3 sentences."},
    ]

    
    
    # Round 1: Each agent gives initial thoughts
    print("--- Round 1: Initial thoughts ---")
    shared_context = "Task: " + task + "\n\n"

    for agent in agents:
        result = ask_agent(agent["name"], agent["prompt"], task)
        agent["last_response"] = result
        shared_context = shared_context + agent["name"] + ": " + result + "\n\n"

    print("\n\n ------ ALL AGENTS OUTPUT ROUND 1 START ------ \n\n", agents , "\n\n ------ ALL AGENTS OUTPUT ROUND 1 END ------ \n\n")
    print("\n\n ------ shared_context ROUND 1 START ------ \n\n", shared_context , "\n\n ------ shared_context ROUND 1 END ------ \n\n")
    
    
    
    # Round 2: Each agent responds to others
    print("--- Round 2: Responding to others ---")
    for agent in agents:
        input_text = shared_context + "\nNow respond to the other perspectives. What do you add or disagree with? 2-3 sentences."
        result = ask_agent(agent["name"], agent["prompt"], input_text)
        agent["last_response"] = result
    
    print("\n\n ------ AGENTS RESPONDING TO OTHERS PERSPECTIVE START ------ \n\n", agents , "\n\n ------ AGENTS RESPONDING TO OTHERS PERSPECTIVE END ------ \n\n")

    
    
    # Combine all perspectives
    summary_input = shared_context + "\nRound 2 responses:\n"
    for agent in agents:
        summary_input = summary_input + agent["name"] + ": " + agent["last_response"] + "\n\n"

    print("\n\n ------ summary_input START ------ \n\n", summary_input , "\n\n ------ summary_input END ------ \n\n")
    
    
    
    # No supervisor - just combine
    final = ask_agent(
        "Synthesizer",
        "Combine all perspectives into a balanced summary. 3-4 sentences.",
        summary_input
    )
    return final

**What happened:** We built the Swarm pattern with 3 equal agents (Optimist, Skeptic, Pragmatist). Round 1: independent thoughts. Round 2: respond to each other. A synthesizer combines everything. No boss.

In [36]:
swarm_pattern("Should a small startup use AI to replace customer support?")

=== SWARM PATTERN ===
Task: Should a small startup use AI to replace customer support?

--- Round 1: Initial thoughts ---
[Optimist]: Using AI for customer support can be a fantastic opportunity for a small startup to enhance efficiency and scalability. It allows for quick responses to customer inquiries, freeing up human agents to 

[Skeptic]: Using AI to replace customer support in a small startup could lead to several issues. While AI can handle basic inquiries efficiently, it might struggle with nuanced or complex customer interactions, 

[Pragmatist]: A small startup can benefit from using AI for customer support by enhancing response times and providing 24/7 assistance, which can improve customer satisfaction. However, it’s crucial to balance auto



 ------ ALL AGENTS OUTPUT ROUND 1 START ------ 

 [{'name': 'Optimist', 'prompt': 'You always see the positive side. Give your perspective in 2-3 sentences.', 'last_response': 'Using AI for customer support can be a fantastic opportu

"Integrating AI into customer support can offer significant benefits for small startups, including efficiency and improved response times, as noted by optimists in the discussion. However, the skeptics highlight potential drawbacks, such as AI's limitations in handling complex customer interactions and the possible erosion of personal relationships. A balanced, pragmatic approach is recommended, where AI handles routine inquiries while ensuring human agents are available for more nuanced issues, thus maintaining customer satisfaction and trust as the startup grows. This hybrid strategy can capitalize on AI's strengths while mitigating the risks of over-reliance on technology."

**What happened:** Three agents with different perspectives tackled the same question. In round 2, they responded to each other's points. The result is a balanced answer from multiple viewpoints.

## 4. Debate Pattern

**What it does:** Two agents take **opposing sides** and argue back and forth. A judge watches and picks the stronger argument.

**When to use it:** When you want to deeply examine both sides of a question. Debate forces each side to address the other's strongest points.

**Steps:**
1. Pro agent argues in favor.
2. Con agent argues against and responds to Pro.
3. They go back and forth for several rounds.
4. A judge reads the full debate and picks the winner.

In [37]:
def debate_pattern(topic):

    pro_prompt = "You argue IN FAVOR of the topic. Be persuasive. 2-3 sentences per round."
    con_prompt = "You argue AGAINST the topic. Be persuasive. 2-3 sentences per round."



    debate_history = "Topic: " + topic + "\n\n"



    for round_num in range(1, 4):
        
        print("--- Round", round_num, "---")

        # Pro side
        pro_input = debate_history + "Give your argument FOR the topic."
        print("\n\n ------ pro_input  ------ \n\n", pro_input, "\n\n ------ pro_input ------ \n\n")

        pro_response = ask_agent("Pro", pro_prompt, pro_input)
        print("\n\n ------ pro_response ------ \n\n", pro_response, "\n\n ------ pro_response ------ \n\n")

        debate_history = debate_history + "Pro (Round " + str(round_num) + "): " + pro_response + "\n\n"
        print("\n\n ------ debate_history ------ \n\n", debate_history, "\n\n ------ debate_history ------ \n\n")

        
        # Con side responds
        con_input = debate_history + "Respond to Pro's argument and give your argument AGAINST."
        print("\n\n ------ con_input ------ \n\n", con_input, "\n\n ------ con_input ------ \n\n")
        
        con_response = ask_agent("Con", con_prompt, con_input)
        print("\n\n ------ con_response ------ \n\n", con_response, "\n\n ------ con_response ------ \n\n")

        debate_history = debate_history + "Con (Round " + str(round_num) + "): " + con_response + "\n\n"
        print("\n\n ------ debate_history ------ \n\n", debate_history, "\n\n ------ debate_history ------ \n\n")

    
    
    
    # Judge decides
    print("--- Judge's Verdict ---")
    judge_prompt = (
        "You are a fair judge. Read the debate and decide which side made stronger arguments. "
        "Explain your decision in 3-4 sentences. State the winner clearly."
    )

    
    verdict = ask_agent("Judge", judge_prompt, debate_history)
    return verdict

**What happened:** We built the Debate pattern with a Pro agent, Con agent, and Judge. They debate for 3 rounds, each responding to the other. This produces deeper analysis than asking one agent.

In [38]:
debate_pattern("Remote work is better than office work for software engineers")

--- Round 1 ---


 ------ pro_input  ------ 

 Topic: Remote work is better than office work for software engineers

Give your argument FOR the topic. 

 ------ pro_input ------ 


[Pro]: Remote work is undeniably better for software engineers as it significantly enhances productivity by allowing them to work in an environment that suits their personal preferences. With fewer distracti



 ------ pro_response ------ 

 Remote work is undeniably better for software engineers as it significantly enhances productivity by allowing them to work in an environment that suits their personal preferences. With fewer distractions and the ability to customize their workspace, engineers can focus better and complete tasks more efficiently. Additionally, remote work eliminates the stress and time wasted on commuting, freeing up valuable hours that can be dedicated to coding, innovation, or even personal interests. 

Moreover, remote work provides access to a global talent pool, enabling companies to

'After evaluating the debate, I find the arguments presented by the Con side to be more compelling. The Con side effectively highlights the importance of in-person collaboration, mentorship, and community, which are critical for nurturing creativity and accountability among software engineers. Furthermore, they address the potential pitfalls of remote work, such as feelings of isolation and blurred boundaries between personal and professional life, emphasizing real challenges that can hinder productivity. While the Pro side makes valid points about flexibility and technology, they do not sufficiently counter the significant benefits of in-person interaction presented by the Con side. Therefore, the winner of this debate is the Con side.'

**What happened:** Two agents debated for 3 rounds, each responding to the other's specific points. The judge read everything and gave a verdict.

## 5. Collaborative Pattern

**What it does:** Agents work **together** on the same output, each adding their expertise. Unlike debate (adversarial), collaborative agents build on each other's work.

**When to use it:** When different parts need different expertise and the final output should combine all expertise seamlessly.

**Steps:**
1. A content expert writes the first draft.
2. A technical reviewer checks and improves accuracy.
3. An editor polishes for clarity.

In [41]:
def collaborative_pattern(task):

    agents = [
        {
            "name": "Content Expert",
            "prompt": "You are a content expert. Write the core content. 3-4 sentences."
        },
        {
            "name": "Technical Reviewer",
            "prompt": "You are a technical reviewer. Check the content for accuracy and add technical details. Keep the original content and add your improvements. 3-4 sentences."
        },
        {
            "name": "Editor",
            "prompt": "You are an editor. Polish the text for clarity and readability. Keep the technical accuracy. 3-4 sentences."
        },
    ]



    current_output = ""

    for agent in agents:
        
        if current_output == "":
            input_text = "Task: " + task
        else:
            input_text = "Task: " + task + "\n\nCurrent draft:\n" + current_output + "\n\nImprove this draft with your expertise."

        
        
        result = ask_agent(agent["name"], agent["prompt"], input_text)
        
        
        print(f"\n\n ------ result of Agent { agent["name"] }------ \n\n", result, f"\n\n ------ result of Agent { agent["name"] }------ \n\n")


        current_output = result

    
    
    print("=== FINAL OUTPUT ===")
    print(current_output)
    return current_output

**What happened:** We built the Collaborative pattern: content expert writes, technical reviewer improves accuracy, editor polishes. Each agent builds on the previous one, like a document going through rounds of review.

In [42]:
collaborative_pattern("Explain how HTTPS encryption works")

[Content Expert]: HTTPS (HyperText Transfer Protocol Secure) encryption works by utilizing a combination of the HyperText Transfer Protocol (HTTP) along with Transport Layer Security (TLS) or its predecessor, Secure So



 ------ result of Agent Content Expert------ 

 HTTPS (HyperText Transfer Protocol Secure) encryption works by utilizing a combination of the HyperText Transfer Protocol (HTTP) along with Transport Layer Security (TLS) or its predecessor, Secure Sockets Layer (SSL). When a user connects to a website via HTTPS, the browser and the server first establish a secure connection through a process called the TLS handshake. This involves the exchange of encryption keys and the negotiation of encryption methods, ensuring that data transmitted between the two parties remains private and secure. Once the secure connection is established, all data transferred is encrypted, protecting it from eavesdropping and tampering during transmission. 

 ------ result of Agent Content Expert-

'HTTPS (HyperText Transfer Protocol Secure) encryption combines the HyperText Transfer Protocol (HTTP) with Transport Layer Security (TLS), or its predecessor, Secure Sockets Layer (SSL). When a user connects to a website via HTTPS, the browser and server establish a secure connection during a process known as the TLS handshake. In this handshake, the server presents its digital certificate, allowing the client to verify its identity, while both parties exchange session keys using asymmetric encryption. This secure negotiation leads to a symmetric encryption session for data transmission, with all information exchanged between the client and server being protected by robust algorithms, such as AES (Advanced Encryption Standard), to prevent eavesdropping and tampering.'

**What happened:** Three agents collaborated on the same text. The content expert created it, the reviewer improved accuracy, and the editor polished it. Each saw and improved the previous agent's work.

## 6. Specialist Routing

**What it does:** A **router** agent reads the question and sends it to the one specialist best suited to answer. Only one specialist handles each question.

**When to use it:** When you have distinct domains (billing, tech support, sales) and each question clearly belongs to one.

**Steps:**
1. A router reads the question.
2. The router picks the best specialist.
3. Only the chosen specialist answers.

In [43]:
def specialist_routing(question):

    specialists = {
        "billing": "You are a billing specialist. Answer billing, payment, and subscription questions. 2-3 sentences.",
        "technical": "You are a technical support specialist. Answer technical, bug, and setup questions. 2-3 sentences.",
        "sales": "You are a sales specialist. Answer questions about products, features, and pricing. 2-3 sentences.",
        "general": "You are a general support agent. Answer general questions. 2-3 sentences."
    }

    
    # Router decides which specialist
    router_prompt = (
        "You are a router. Read the customer question and decide which specialist should handle it. "
        "Options: billing, technical, sales, general. "
        "Reply with ONLY one word: the specialist name."
    )

    route = ask_agent("Router", router_prompt, question, max_tokens=10)
    chosen = route.strip().lower()



    # Clean up the response
    for specialist_name in specialists:
        if specialist_name in chosen:
            chosen = specialist_name
            break


    if chosen not in specialists:
        chosen = "general"



    print("Routed to:", chosen)
    print()

    
    
    # Specialist answers
    answer = ask_agent(
        chosen.capitalize() + " Specialist",
        specialists[chosen],
        question
    )

    
    return answer

**What happened:** We built Specialist Routing with a router and 4 specialists (billing, technical, sales, general). The router picks one specialist per question. Only one runs, making it efficient.

In [44]:
specialist_routing("I was charged twice for my subscription last month")

[Router]: billing

Routed to: billing

[Billing Specialist]: I apologize for the inconvenience! Please check your billing statement to confirm the charges. If you see two charges and believe this is an error, please contact customer support with your account de



'I apologize for the inconvenience! Please check your billing statement to confirm the charges. If you see two charges and believe this is an error, please contact customer support with your account details so we can investigate and resolve the issue promptly.'

**What happened:** The router read the billing question and sent it to the billing specialist.

In [45]:
specialist_routing("How do I install your Python SDK?")

[Router]: technical

Routed to: technical

[Technical Specialist]: To install the Python SDK, you can use pip, the package installer for Python. Run the command `pip install your-sdk-name` in your terminal or command prompt, replacing "your-sdk-name" with the actual 



'To install the Python SDK, you can use pip, the package installer for Python. Run the command `pip install your-sdk-name` in your terminal or command prompt, replacing "your-sdk-name" with the actual name of the SDK. If you need further configuration, refer to the official documentation provided with the SDK for additional setup instructions.'

**What happened:** The router sent billing questions to billing and technical questions to technical. Each question went to exactly one expert.

## 7. Consensus Pattern

**What it does:** Multiple agents independently answer the same question. A consensus checker compares their answers to find agreement and flag disagreements.

**When to use it:** For high-stakes decisions where you want confidence. Multiple independent opinions reduce the chance of error.

**Steps:**
1. Multiple agents answer the same question independently.
2. A consensus checker compares all answers.
3. The checker reports agreements and disagreements.